In [ ]:
#To run on scvi_env environement

import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
os.environ["OMP_NUM_THREADS"]      = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"]      = "1"
os.environ["NUMEXPR_NUM_THREADS"]  = "1"

import random, warnings
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import scvi
warnings.filterwarnings("ignore")
sc.logging.print_header()
scvi.settings.seed = 0

In [ ]:
adata = sc.read_h5ad("/path/to/your/file.h5ad")


In [ ]:
SAVE_PATH  = "/path/to/output/subset_integrated.h5ad"
MODEL_DIR  = "/path/to/output/scvi_model/"

PARAMS = dict(
    seed               = 1234,
    batch_key          = "section",

    # ── scVI ────────────────────────────────────────────────────────────────
    device             = "cpu",  # "mps" for Apple GPU, "cpu" for CPU
    n_latent           = 15,     # fewer dims than full dataset (default 20)
    n_layers           = 2,
    max_epochs         = 500,
    early_stopping     = True,
    n_train_cells      = None,   # None = train on all cells (suitable for subsets)
    batch_size         = 512,

    # ── Clustering ──────────────────────────────────────────────────────────
    n_neighbors        = 30,
    leiden_resolutions = [0.3, 0.5, 0.8, 1.0, 1.5],
    umap_min_dist      = 0.1,
)


### 3.5 · Run scVI

In [ ]:
def run_scvi(adata):
    """
    Train scVI on a stratified subsample (faster + less memory),
    then embed all cells using the trained model.

    - Subsamples ~n_train_cells cells stratified by section
    - device = 'mps' for Apple GPU, 'cpu' for CPU (set in PARAMS)
    - Embeds ALL cells after training
    Adds obsm['X_scVI'] to adata.
    Returns the trained model.
    """
    import torch
    from unittest.mock import patch
    from lightning.pytorch.accelerators import MPSAccelerator

    # ── Device setup ────────────────────────────────────────────────────────
    device = PARAMS["device"]
    if device == "mps":
        if not torch.backends.mps.is_available():
            print("  MPS not available — falling back to CPU")
            device = "cpu"
        else:
            scvi.settings.dl_num_workers = 0
            print("  Using MPS (Apple Silicon GPU)")
    if device == "cpu":
        print("  Using CPU")

    # ── Set X = raw counts ───────────────────────────────────────────────────
    if "counts" in adata.layers:
        adata.X = adata.layers["counts"].copy()

    # ── Train on all cells or subsample ─────────────────────────────────────
    if PARAMS.get("n_train_cells") is None:
        adata_train = adata.copy()
        print(f"  Training on ALL {adata_train.n_obs:,} cells "
              f"({adata_train.obs[PARAMS['batch_key']].nunique()} sections)...")
    else:
        n_train   = PARAMS["n_train_cells"]
        n_per_sec = max(1, n_train // adata.obs[PARAMS["batch_key"]].nunique())
        train_pos = []
        for sec, grp in adata.obs.groupby(PARAMS["batch_key"]):
            n_sample = min(len(grp), n_per_sec)
            sampled  = grp.sample(n_sample, random_state=PARAMS["seed"])
            train_pos.extend(adata.obs_names.get_indexer(sampled.index))
        adata_train = adata[train_pos].copy()
        print(f"  Training on {adata_train.n_obs:,} cells "
              f"({adata_train.obs[PARAMS['batch_key']].nunique()} sections)...")

    # ── Build model ──────────────────────────────────────────────────────────
    scvi.model.SCVI.setup_anndata(adata_train, batch_key=PARAMS["batch_key"])
    model = scvi.model.SCVI(
        adata_train,
        n_latent = PARAMS["n_latent"],
        n_layers = PARAMS["n_layers"],
    )

    # ── Train ────────────────────────────────────────────────────────────────
    def _do_train(acc, dev):
        model.train(
            max_epochs     = PARAMS["max_epochs"],
            early_stopping = PARAMS["early_stopping"],
            batch_size     = PARAMS["batch_size"],
            accelerator    = acc,
            devices        = dev,
        )

    if device == "mps":
        try:
            with patch.object(MPSAccelerator, "is_available", return_value=True), \
                 patch("lightning.fabric.utilities.device_parser._get_all_available_gpus",
                       return_value=[0]):
                _do_train("mps", 1)
        except Exception as e:
            print(f"  MPS failed ({e}) — falling back to CPU")
            _do_train("cpu", 1)
    else:
        _do_train("cpu", 1)

    # ── Embed ALL cells ──────────────────────────────────────────────────────
    print(f"  Embedding all {adata.n_obs:,} cells...")
    scvi.model.SCVI.setup_anndata(adata, batch_key=PARAMS["batch_key"])
    adata.obsm["X_scVI"] = model.get_latent_representation(adata)
    print(f"  Done — latent: {adata.obsm['X_scVI'].shape}")

    return model

### 3.6 · Cluster & UMAP

In [ ]:
def cluster_and_embed(adata):
    """
    Build neighbourhood graph on X_scVI, compute UMAP and Leiden clusters
    at multiple resolutions.
    Adds:
      obsm["X_umap_scVI"]
      obs["leiden_<resolution>"] for each resolution in leiden_resolutions
    """
    sc.pp.neighbors(adata, use_rep="X_scVI", n_neighbors=PARAMS["n_neighbors"])
    sc.tl.umap(adata, min_dist=PARAMS["umap_min_dist"])
    adata.obsm["X_umap_scVI"] = adata.obsm["X_umap"].copy()

    leiden_cols = []
    for r in PARAMS["leiden_resolutions"]:
        col = f"leiden_{r:g}"
        sc.tl.leiden(adata, resolution=float(r), key_added=col)
        n_cl = adata.obs[col].nunique()
        print(f"  resolution={r} → {n_cl} clusters")
        leiden_cols.append(col)

    sc.pl.umap(adata, color=leiden_cols + ["section"], ncols=3)


### 3.7 · Save

In [ ]:
def save_integrated(adata, save_path):
    """
    Restore adata.X = transformed before saving.
    All layers and embeddings are preserved.
    """
    os.makedirs(os.path.dirname(save_path), exist_ok=True)

    if "transformed" in adata.layers:
        adata.X = adata.layers["transformed"].copy()
        if sp.issparse(adata.X):
            adata.X = adata.X.tocsr()

    adata.write(save_path)
    print(f"  [✓] Saved → {save_path}")
    print(f"      {adata.shape[0]:,} cells x {adata.shape[1]:,} genes")
    print(f"      layers : {list(adata.layers.keys())}")
    print(f"      obsm   : {list(adata.obsm.keys())}")
    print(f"      obs    : {list(adata.obs.columns)}")


## 4 · Run

In [ ]:
print("[1/3] scVI integration...")
model_scvi = run_scvi(adata)

print("\n[2/3] Clustering & UMAP...")
cluster_and_embed(adata)

print("\n[3/3] Saving...")
model_scvi.save(MODEL_DIR, overwrite=True)
print(f"  [✓] Model saved → {MODEL_DIR}")
save_integrated(adata, SAVE_PATH)
print("Done!")


In [ ]:
def plot_training_history(model):
    history = model.history

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    axes[0].plot(history["elbo_train"],      label="train",      linewidth=1.5)
    axes[0].plot(history["elbo_validation"], label="validation", linewidth=1.5, ls="--")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("ELBO loss")
    axes[0].set_title("Training convergence")
    axes[0].legend()

    axes[1].plot(history["reconstruction_loss_train"],      label="train",      linewidth=1.5)
    axes[1].plot(history["reconstruction_loss_validation"], label="validation", linewidth=1.5, ls="--")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Reconstruction loss")
    axes[1].set_title("Reconstruction loss")
    axes[1].legend()

    n_epochs = len(history["elbo_train"])
    plt.suptitle(f"scVI training — stopped at epoch {n_epochs}")
    plt.tight_layout()
    plt.show()

    print(f"  Epochs run      : {n_epochs}")
    print(f"  Final train ELBO: {float(history['elbo_train'].values[-1]):.2f}")
    print(f"  Final val ELBO  : {float(history['elbo_validation'].values[-1]):.2f}")

plot_training_history(model_scvi)